<H1> First project: charged particle trapped in a magnetic field </H1>

**By:** Santiago Andrés Acosta Díaz

**For:** Computational Astrophysics

### Some Markdown/LaTeX commands
In this block, I'm writing some commands to ease the LaTeX writing process.

_Ignore this cell_


$$
\newcommand{\pdv}[2]{\frac{\partial #1}{\partial #2}}
\newcommand{\vdot}[1]{{\dot{\vec #1}}}
\newcommand{\vddot}[1]{{\ddot{\vec #1}}}
$$

### Python library imports


In [12]:

import numpy as np
from numba import jit
import matplotlib.pyplot as plt

# Phase 1

<H3> Mathematical framing of Particle Dynamics </H3>

## Derivation of the equations of motion

In this section, we'll analytically derive the motion equations for the Cartesian coordinates of a charged particle under the influence of a dipole magnetic field, given the Lorentz Force equation

$$
F_{L} = q \left( \vec E + \vec v \times \vec B \right)
$$


### Solution

For this, we first have to state that 

* Electric field $\vec E = 0$
* The magnetic field is a dipole with shape (given from eq (6) from the study material) $\vec B_{dip} = -\frac{B_0 R_e^3}{r^5} \left[ 3xz \hat x + 3yz \hat y + (2z^2 - x^2 - y^2)\hat z \right]$

Now, with that, we have the equation to solve as follows

$$ 
\begin{align}
\vddot r &= \frac{q}{m} \dot {\vec r} \times \vec B_{dip} \\
\vddot r &= \underbrace{-\frac{q B_0 R_e^3}{m}}_{\Omega} \frac{1}{r^5}\, \dot {\vec r}  \times (3xz, 3yz, 3z^2 - r^2)
\end{align}
$$

considering that $r^2 = x^2 + y^2 + z^2$ and we define $\Omega = -\frac{q B_0 R_e^3}{m}$ to ease the notation, as it is a global constant.

Now, as numerically we have to treat each coordinate separately, we'll have to find the vector expression of the cross product

$$
\begin{align}
    \vdot r \times (3xz, 3yz, 3z^2 - r^2) & = 
    \left|
    \begin{matrix}
        \hat x & \hat y & \hat z \\ 
        \dot x & \dot y & \dot z \\ 
        3xz & 3yz & 3z^2 - r^2
    \end{matrix} 
    \right| \\
    &= \left( \dot y (3z^2 - r^2) - \dot z (3yz),\, \dot z(3xz) - \dot x(3z^2 - r^2),\, \dot x (3yz) - \dot y (3xz) \right)
\end{align}
$$

from where we can get the equations of motion for each coordinate



### Answer

$$
\boxed{
\begin{gather}
    \ddot x  = \frac{\Omega}{r^5} \left[ (3 z^2 - r^2) \dot y - 3yz \dot z \right] \\
    \ddot y  = \frac{\Omega}{r^5} \left[ 3xz \dot z - (3 z^2 - r^2) \dot x \right] \\
    \ddot z  = \frac{\Omega}{r^5} \left[ 3z \left( y \dot x  - x \dot y \right) \right]
\end{gather}}
$$

## Numerical differentiation specification

Implement finite difference approximations derived from Lagrange interpolation.

In this section, we're asked to implement finite differences derivation algorithms from Lagrange interpolation according to "A General Formula for the Numerical Differentiation of a Function" by A. K. SINGH AND G. R. THORPE. What matters here is that these formulas are the same ones we get from the Taylor expansions done previously during class, as one can see on equations (3.2) and (3.3) of the paper. 

With that in mind, the task is no other than implement these algorithms: 


For **first derivative with $n=2$:**

* Forwards difference: $$ f'(x) \approx \frac{4f(x + h) - f(x + 2h) - 3f(x)}{2h}$$
* Backwards difference: $$ f'(x) \approx \frac{3f(x) + f(x - 2h) - 4f(x - h)}{2h}$$
* centered difference: $$ f'(x) \approx \frac{f(x + h) - f(x-h)}{2h}$$


For **first derivative with $n=3$:** the same as we have on the table in the study material for $s = 4$



For **second derivative with $n=2$:**

* Forwards difference: $$ f''(x) \approx \frac{f(x - h) - 2f(x) + f(x+h)}{h^2}$$

In [36]:
'''
In order to make the calculations of the first derivative efficient and only needing one function, 
we'll create a look-up table that contains all the coefficients for the f_i f_{\pm i} and f_{\pm 2i} algorithms

STRUCTURE: dict
keys: tuple (nodal points n, type of algorithm)
(n = [2, 3], type of algorithm = ["f(orward)", "b(ackward)", "c(entral)"]) 

items: tuple
( (array with the coefficients), (array with the offests) )
'''

FD_COEFFS = {
    (2, "f"): (np.array([-3/2, 2, -1/2]), np.array([0, 1, 2])),
    (2, "b"): (np.array([3/2, -2, 1/2]), np.array([0, -1, -2])),
    (2, "c"): (np.array([-1/2, 0, 1/2]), np.array([-1, 0, 1])),
    
    (3, "f"): (np.array([-11, 18, -9, 2]) / 6.0, np.array([0, 1, 2, 3])),
    (3, "b"): (np.array([-2, 9, -18, 11]) / 6.0, np.array([-3, -2, -1, 0])),
    (3, "c1"): (np.array([-2, -3, 6, -1]) / 6.0, np.array([-1, 0, 1, 2])),
    (3, "c2"): (np.array([1, -6, 3, 2]) / 6.0, np.array([-2, 1, 0, 1]))
}


def fo_derivative(f, x, h=1e-5, n=3, t="c"):
    """ This function represents the first order derivative approximation.
    It basically performs the algorithms seen above but, in order to have them all in one
    only function and still be efficient, we use a lookup table with the coefficients and the offsets """

    # We get the coefficients and offsets array
    coeffs, offsets = FD_COEFFS[(n, t)]
    
    # Create shifted evaluation points  
    shifted = x[np.newaxis, :] + offsets[:, np.newaxis] * h
    values = f(shifted)   
    
    return (coeffs[:, None] * values).sum(axis=0) / h

def so_derivative(f, x, h=1e-5):
    """ Now, for the second order derivative, which we need a single algorithm """

    return (f(x - h) - 2.0 * f(x) + f(x + h)) / (h**2)

# Phase 2

<H3>Algoritmic generalization and lagrange interpolation</H3>